# Multimodal

In [1]:
# Load library
from transformers import pipeline

# Create a Image text to Text
A multimodal model that takes an image and a prompt as input and outputs a text response

## Use cases
* Visual Questin and Answer - Ask a question about an image
* Document understanding - Analyze charts, or get details from receipts

In [2]:
image_answering = pipeline("image-text-to-text")


[transformers] No model was supplied, defaulted to Qwen/Qwen3-VL-2B-Instruct and revision 8964489.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

## Perform an inference from the created model

In [3]:
image = "https://www.sheknows.com/wp-content/uploads/2018/08/FlickrTimDorrDogButtSniff_wu3qep.jpeg"
text = "The dogs are "
#image = "https://static.boredpanda.com/blog/wp-content/uploads/2016/09/mother-bear-cubs-animal-parenting-21-57e3a2161d7f7__880.jpg"
#text = "Where is the baby bear?"

image_answering(text=text, image=image, max_new_tokens=512)


[transformers] The input data was not formatted as a chat with dicts containing 'role' and 'content' keys, even though this model supports chat. Consider using the chat format for better results. For more information, see https://huggingface.co/docs/transformers/en/chat_templating
[transformers] Keyword argument `image` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [4]:

# Import libraries
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoProcessor



In [5]:
# Load model, and processor
model_name = "Qwen/Qwen3-VL-2B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForImageTextToText.from_pretrained(model_name)


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

## Chat templates

Chat templates are structured formatting schemes that define how conversations between users and language models are organized. They play a crucial role in ensuring the model correctly interprets the different roles and turns in a conversation.

LLMs are fundamentally trained on sequences of tokens. Without a template, the model would have no way to distinguish between what the user said, what the system prompt instructs. Chat templates solve this by injecting special tokens or formatting conventions that the model was trained to recognize.

Every chat-tuned model was fine-tuned on a specific text format. LLaMA expects something like `[INST] ... [/INST]`, ChatML uses `<|im_start|>user\n...<|im_end|>`, Qwen has its own variant, and so on. If you feed the model text in the wrong format, it still generates something, but quality collapses because the model never saw that pattern during training.

In [7]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {%- if messages[0].content is string %}
            {{- messages[0].content }}
        {%- else %}
            {%- for content in messages[0].content %}
                {%- if 'text' in content %}
                    {{- content.text }}
                {%- endif %}
            {%- endfor %}
        {%- endif %}
        {{- '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if me

## Structured input

Create a structured input and convert it to the model's chat template



In [31]:
# Create structured input
image = "https://www.sheknows.com/wp-content/uploads/2018/08/FlickrTimDorrDogButtSniff_wu3qep.jpeg"
text = "The dogs are "
image = "https://static.boredpanda.com/blog/wp-content/uploads/2016/09/mother-bear-cubs-animal-parenting-21-57e3a2161d7f7__880.jpg"
text = "What is the colour of the bear?"



# Generate Prompt

Create the prompt with `apply_chat_template`.
* add_generation_prompt - append a special sequence of tokens to the end to instruct the model to generate response
* tokenize - tokenize the generated prompt
* return_dict - return the output as a dictionary vs a single tensor eg. `input_ids`, `attention_mask`, `pixel_values`
* return_tensors - pt (PyTorch), tf (TensorFlow)

In [32]:
image = "https://www.sheknows.com/wp-content/uploads/2018/08/FlickrTimDorrDogButtSniff_wu3qep.jpeg"
text = "The dogs are "
image = "https://static.boredpanda.com/blog/wp-content/uploads/2016/09/mother-bear-cubs-animal-parenting-21-57e3a2161d7f7__880.jpg"
text = "What is the color of the baby bear?"

message = [
    # {
    #   "role": "system",
    #   "content": [
    #       {
    #         "type": "text",
    #         "text": "You are a helpful assistant. Do not answer an inappropriate questons. Inappropriate means offensive, vile or pornographic."
    #       }
    #   ]
    # },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": text
            },
            {
                "type": "image",
                "url": "image"
            }
        ]
    }
]

## Dump the output
* input_ids - the tokens
* attention_mask - which token to attend, typically used to identify padding in batch
* mm_token_type_ids - identify the token type in the multimodal prompt
* pixel_values - the processed image data
* image_grid_thw - Time, Height, Width describes how the image are divided into patches or tiles for processing

In [33]:
inputs = tokenizer.apply_chat_template(
    message,
    tokenizer=True,
    return_tensors='pt',
    return_dict=True,
    add_generation_prompt=True
)

inputs


{'input_ids': tensor([[151644,    872,    198,   3838,    374,    279,   1894,    315,    279,
           8770,  11722,     30, 151652, 151655, 151653, 151645,    198, 151644,
          77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

# Inference

Generate response with the prompt from the model and decode the output

In [36]:
outputs = model.generate(do_sample=True, temperature=0.3, **inputs)
print(outputs)

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1612: UserWarning: Using the model-agnostic default `max_length` (=40) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


tensor([[151644,    872,    198,   3838,    374,    279,   1894,    315,    279,
           8770,  11722,     30, 151652, 151655, 151653, 151645,    198, 151644,
          77091,    198,  28715,    389,    279,   2168,   3897,     11,    279,
           8770,  11722,    374,   4158,     13, 151645]])


In [37]:
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(output_text)

user
What is the color of the baby bear?
assistant
Based on the image provided, the baby bear is white.
